In [1]:
import torch, platform

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

Python: 3.12.13
PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
GPU memory GB: 15.64


In [2]:
!pip install -q transformers datasets peft trl bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 19.4 MB/s eta 0:00:00


In [3]:
import torch

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)

from peft import (
    LoraConfig,
    get_peft_model,
)

from trl import SFTTrainer
from datasets import Dataset

print("All imports successful.")

All imports successful.


In [4]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizer loaded.")
print("Pad token:", tokenizer.pad_token)
print("EOS token:", tokenizer.eos_token)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded.
Pad token: <|endoftext|>
EOS token: <|im_end|>


In [5]:
raw_examples = [
    {
        "question": "Why is HBM bandwidth important for AI inference?",
        "answer": "HBM bandwidth matters because inference repeatedly moves weights, activations, and KV cache data between memory and compute. If bandwidth is insufficient, GPU compute can sit idle and latency increases."
    },
    {
        "question": "What is KV cache in transformers?",
        "answer": "KV cache stores previously computed attention keys and values during autoregressive generation. This avoids recomputing attention history for every new token, reducing latency and compute cost."
    },
    {
        "question": "Why does P99 latency matter in AI systems?",
        "answer": "P99 latency captures tail user experience. Even if average latency looks healthy, high P99 can make an AI product feel unreliable for real users."
    },
    {
        "question": "Why use LoRA instead of full fine-tuning?",
        "answer": "LoRA updates small low-rank adapter matrices instead of all model weights. This reduces GPU memory, training cost, and iteration time while preserving most base model capability."
    },
    {
        "question": "What problem does QLoRA solve?",
        "answer": "QLoRA makes fine-tuning cheaper by loading the frozen base model in 4-bit quantized form while training small LoRA adapters. This allows experimentation on constrained GPUs."
    },
    {
        "question": "Why is training loss not enough to judge fine-tuning quality?",
        "answer": "Training loss only measures how well the model predicts tokens in the training data. It does not guarantee factual accuracy, reasoning quality, or good behavior on unseen prompts."
    },
    {
        "question": "When is RAG better than fine-tuning?",
        "answer": "RAG is better when the answer depends on factual, changing, or enterprise-specific knowledge. Fine-tuning is better for behavior, style, formatting, and lightweight specialization."
    },
    {
        "question": "Why can small fine-tuning datasets be risky?",
        "answer": "Small datasets can cause the model to learn surface patterns, vocabulary, or formatting without learning robust concepts. They also increase overfitting risk."
    },
    {
        "question": "What is the PM tradeoff between model quality and inference latency?",
        "answer": "Higher quality models often require more compute, memory, and latency. A PM must balance answer quality against user experience, cost per request, and infrastructure capacity."
    },
    {
        "question": "Why does quantization reduce GPU memory usage?",
        "answer": "Quantization stores model weights using fewer bits, such as 4-bit instead of 16-bit. This lowers memory footprint and can make larger models fit on smaller GPUs."
    },
    {
        "question": "What is gradient accumulation?",
        "answer": "Gradient accumulation collects gradients across multiple small batches before applying an optimizer step. It simulates a larger batch size when GPU memory is limited."
    },
    {
        "question": "Why does tokenizer alignment matter in instruction tuning?",
        "answer": "Instruction-tuned models expect prompts in a specific chat format. If the dataset format does not match the tokenizer chat template, generation quality can degrade even when loss improves."
    },
    {
        "question": "What is adapter-based deployment?",
        "answer": "Adapter-based deployment keeps the base model fixed and loads small task- or customer-specific adapter weights. This reduces storage cost and supports faster customization."
    },
    {
        "question": "Why is P50 latency not enough for AI product monitoring?",
        "answer": "P50 latency only shows the median experience. AI products often fail at the tail, so P95 and P99 are needed to detect slow responses that affect real users."
    },
    {
        "question": "Why does KV cache increase memory pressure during long-context inference?",
        "answer": "KV cache grows with sequence length because the model stores keys and values for previous tokens. Longer conversations require more memory, which can limit batch size and throughput."
    },
    {
        "question": "What is the difference between HBM capacity and HBM bandwidth?",
        "answer": "HBM capacity determines how much data can fit near the GPU, while HBM bandwidth determines how quickly that data can be moved. Both affect LLM serving performance."
    },
    {
        "question": "Why is defect classification a good fit for ML in semiconductor manufacturing?",
        "answer": "Defect classification can use image patterns from inspection data to flag likely failures earlier. The product value comes from higher recall, faster qualification, and reduced manual review burden."
    },
    {
        "question": "What should an AI infra PM monitor after deploying a fine-tuned model?",
        "answer": "An AI infra PM should monitor latency, cost per request, GPU utilization, quality metrics, hallucination rate, fallback rate, and drift in user queries or model behavior."
    },
    {
        "question": "Why can adapter files be useful in production?",
        "answer": "Adapter files are small compared to full model checkpoints. They can be versioned, swapped, and deployed per use case without duplicating the full base model."
    },
    {
        "question": "What is the main infrastructure benefit of PEFT?",
        "answer": "PEFT reduces the number of trainable parameters, which lowers memory usage and training cost. This enables faster experimentation without full model retraining."
    },
]

formatted_examples = []

for ex in raw_examples:
    messages = [
        {
            "role": "system",
            "content": "You are a concise AI infrastructure and semiconductor product expert. Answer like a senior technical PM."
        },
        {
            "role": "user",
            "content": ex["question"]
        },
        {
            "role": "assistant",
            "content": ex["answer"]
        },
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    formatted_examples.append({"text": text})

dataset = Dataset.from_list(formatted_examples)

print(dataset)
print(dataset[0]["text"])

Dataset({
    features: ['text'],
    num_rows: 20
})
<|im_start|>system
You are a concise AI infrastructure and semiconductor product expert. Answer like a senior technical PM.<|im_end|>
<|im_start|>user
Why is HBM bandwidth important for AI inference?<|im_end|>
<|im_start|>assistant
HBM bandwidth matters because inference repeatedly moves weights, activations, and KV cache data between memory and compute. If bandwidth is insufficient, GPU compute can sit idle and latency increases.<|im_end|>



In [6]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

print(bnb_config)

BitsAndBytesConfig {
  "_load_in_4bit": true,
  "_load_in_8bit": false,
  "bnb_4bit_compute_dtype": "float16",
  "bnb_4bit_quant_storage": "uint8",
  "bnb_4bit_quant_type": "nf4",
  "bnb_4bit_use_double_quant": false,
  "llm_int8_enable_fp32_cpu_offload": false,
  "llm_int8_has_fp16_weight": false,
  "llm_int8_skip_modules": null,
  "llm_int8_threshold": 6.0,
  "load_in_4bit": true,
  "load_in_8bit": false,
  "quant_method": "bitsandbytes"
}



In [7]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

print("Model loaded.")
print("Device:", model.device)

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded.
Device: cuda:0


In [8]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093


In [9]:
training_args = TrainingArguments(
    output_dir="./qwen-day22-lora",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=5,
    learning_rate=2e-4,
    fp16=True,
    bf16=False,
    logging_steps=1,
    save_strategy="no",
    report_to="none",
)

print(training_args)

TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=False,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.NO,
eval_use_gather_object=False,

In [10]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)

print("Trainer created.")

Adding EOS to train dataset:   0%|          | 0/20 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/20 [00:00<?, ? examples/s]

Trainer created.


In [12]:
from collections import Counter

dtype_counts = Counter()

for name, param in model.named_parameters():
    if param.requires_grad:
        dtype_counts[str(param.dtype)] += param.numel()

print(dtype_counts)

for name, param in model.named_parameters():
    if param.requires_grad:
        print(name, param.dtype, param.shape)
        break

Counter({'torch.bfloat16': 540672})
base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight torch.bfloat16 torch.Size([8, 896])


In [13]:
for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.float()

from collections import Counter

dtype_counts = Counter()
for name, param in model.named_parameters():
    if param.requires_grad:
        dtype_counts[str(param.dtype)] += param.numel()

print(dtype_counts)

Counter({'torch.float32': 540672})


In [14]:
trainer.train()

Step,Training Loss
1,5.651494
2,5.680740
3,5.539491
4,5.249287
5,4.765998
6,4.680449
7,4.418176
8,4.824560
9,4.270272
10,4.083686


TrainOutput(global_step=25, training_loss=4.180382223129272, metrics={'train_runtime': 39.3229, 'train_samples_per_second': 2.543, 'train_steps_per_second': 0.636, 'total_flos': 17091078098688.0, 'train_loss': 4.180382223129272})

In [15]:
adapter_dir = "./qwen-day22-lora-adapter-20examples"

model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)

print(f"Saved adapter to: {adapter_dir}")
!ls -lh {adapter_dir}

Saved adapter to: ./qwen-day22-lora-adapter-20examples
total 13M
-rw-r--r-- 1 root root 1.1K May 20 04:59 adapter_config.json
-rw-r--r-- 1 root root 2.1M May 20 04:59 adapter_model.safetensors
-rw-r--r-- 1 root root 2.5K May 20 04:59 chat_template.jinja
-rw-r--r-- 1 root root 5.1K May 20 04:59 README.md
-rw-r--r-- 1 root root  665 May 20 04:59 tokenizer_config.json
-rw-r--r-- 1 root root  11M May 20 04:59 tokenizer.json


In [16]:
eval_prompts = [
    "What is KV cache in transformers?",
    "Why does P99 latency matter for AI products?",
    "When should a team use RAG instead of fine-tuning?",
    "Why does HBM bandwidth matter for LLM inference?",
    "What should an AI infra PM monitor after deploying a fine-tuned model?",
]

print(f"Eval prompts: {len(eval_prompts)}")
for i, prompt in enumerate(eval_prompts, 1):
    print(f"{i}. {prompt}")

Eval prompts: 5
1. What is KV cache in transformers?
2. Why does P99 latency matter for AI products?
3. When should a team use RAG instead of fine-tuning?
4. Why does HBM bandwidth matter for LLM inference?
5. What should an AI infra PM monitor after deploying a fine-tuned model?


In [17]:
from transformers import AutoModelForCausalLM

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

print("Base model loaded.")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Base model loaded.


In [18]:
def generate_response(model_to_use, prompt, max_new_tokens=120):
    messages = [
        {
            "role": "system",
            "content": "You are a concise AI infrastructure and semiconductor product expert. Answer like a senior technical PM."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(text, return_tensors="pt").to(model_to_use.device)

    outputs = model_to_use.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Return only assistant response
    if "assistant" in decoded:
        return decoded.split("assistant")[-1].strip()

    return decoded.strip()

print("Generation helper ready.")

Generation helper ready.


In [19]:
eval_rows = []

for prompt in eval_prompts:
    base_output = generate_response(base_model, prompt)
    tuned_output = generate_response(model, prompt)

    eval_rows.append({
        "prompt": prompt,
        "base_output": base_output,
        "fine_tuned_output": tuned_output,
    })

print(f"Generated outputs for {len(eval_rows)} prompts.")

for row in eval_rows:
    print("\n" + "="*80)
    print("PROMPT:", row["prompt"])
    print("\nBASE MODEL:\n", row["base_output"])
    print("\nFINE-TUNED MODEL:\n", row["fine_tuned_output"])

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Generated outputs for 5 prompts.

PROMPT: What is KV cache in transformers?

BASE MODEL:
 KV Cache (Key-Value) Cache in Transformers refers to the specialized memory used by the Transformer architecture, particularly within the transformer model's memory graph. It serves as an efficient way to store key-value pairs across multiple layers of the network.

Key Points:
1. Purpose: To efficiently manage large amounts of data that can be stored in memory.
2. Architecture: Typically includes two types of caches:
   - **KV Caches**: Used for storing keys (e.g., attention weights).
   - **Value Caches**: Used for storing values (e.g., positional embeddings).
3. Key Characteristics

FINE-TUNED MODEL:
 KV Cache (Key-Value) Cache is an internal data structure used by Transformers to store key-value pairs during training, enabling efficient retrieval of model parameters for inference.

PROMPT: Why does P99 latency matter for AI products?

BASE MODEL:
 P99 latency is crucial in the context of AI pr

In [20]:
evaluation_rules = {
    "What is KV cache in transformers?": [
        "attention",
        "keys",
        "values",
    ],
    "Why does P99 latency matter for AI products?": [
        "latency",
        "user",
        "tail",
    ],
    "When should a team use RAG instead of fine-tuning?": [
        "retrieval",
        "knowledge",
        "factual",
    ],
    "Why does HBM bandwidth matter for LLM inference?": [
        "memory",
        "bandwidth",
        "latency",
    ],
    "What should an AI infra PM monitor after deploying a fine-tuned model?": [
        "latency",
        "cost",
        "hallucination",
    ],
}

def keyword_score(text, keywords):
    text = text.lower()
    return sum(1 for keyword in keywords if keyword.lower() in text)

for row in eval_rows:
    prompt = row["prompt"]
    keywords = evaluation_rules[prompt]

    base_score = keyword_score(row["base_output"], keywords)
    tuned_score = keyword_score(row["fine_tuned_output"], keywords)

    row["base_score"] = base_score
    row["tuned_score"] = tuned_score

print("Evaluation complete.\n")

for row in eval_rows:
    print("="*80)
    print("PROMPT:", row["prompt"])
    print(f"BASE SCORE: {row['base_score']}")
    print(f"TUNED SCORE: {row['tuned_score']}")

Evaluation complete.

PROMPT: What is KV cache in transformers?
BASE SCORE: 3
TUNED SCORE: 0
PROMPT: Why does P99 latency matter for AI products?
BASE SCORE: 1
TUNED SCORE: 1
PROMPT: When should a team use RAG instead of fine-tuning?
BASE SCORE: 0
TUNED SCORE: 0
PROMPT: Why does HBM bandwidth matter for LLM inference?
BASE SCORE: 3
TUNED SCORE: 3
PROMPT: What should an AI infra PM monitor after deploying a fine-tuned model?
BASE SCORE: 0
TUNED SCORE: 0


In [21]:
import pandas as pd

eval_df = pd.DataFrame([
    {
        "Prompt": row["prompt"],
        "Base Score": row["base_score"],
        "Fine-Tuned Score": row["tuned_score"],
    }
    for row in eval_rows
])

eval_df

,Prompt,Base Score,Fine-Tuned Score
0,What is KV cache in transformers?,3,0
1,Why does P99 latency matter for AI products?,1,1
2,When should a team use RAG instead of fine-tun...,0,0
3,Why does HBM bandwidth matter for LLM inference?,3,3
4,What should an AI infra PM monitor after deplo...,0,0


In [23]:
memory_cost_summary = {
    "Base model": "Qwen2.5-0.5B-Instruct",
    "GPU": "Tesla T4",
    "Trainable parameters": "540,672",
    "Total parameters": "494,573,440",
    "Trainable %": "0.1093%",
    "Peak allocated VRAM": "~1.08 GB",
    "Peak reserved VRAM": "~1.53 GB",
    "Adapter size": "~2.1 MB",
    "Training runtime": "~39 sec",
    "Training examples": "20",
}

pd.DataFrame(memory_cost_summary.items(), columns=["Metric", "Value"])

,Metric,Value
0,Base model,Qwen2.5-0.5B-Instruct
1,GPU,Tesla T4
2,Trainable parameters,"540,672"
3,Total parameters,"494,573,440"
4,Trainable %,0.1093%
5,Peak allocated VRAM,~1.08 GB
6,Peak reserved VRAM,~1.53 GB
7,Adapter size,~2.1 MB
8,Training runtime,~39 sec
9,Training examples,20
